In [11]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 1. DEFINE MODEL ARCHITECTURE
class WaferCNN(nn.Module):
    def __init__(self, in_ch=1, num_classes=8, base_ch=32, dropout=0.3):
        super().__init__()
        def conv_bn_relu(in_c, out_c, k=3):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, k, padding=k // 2, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
            )
        self.block1 = nn.Sequential(
            conv_bn_relu(in_ch, base_ch),
            conv_bn_relu(base_ch, base_ch),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout * 0.5),
        )
        self.block2 = nn.Sequential(
            conv_bn_relu(base_ch, base_ch * 2),
            conv_bn_relu(base_ch * 2, base_ch * 2),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout * 0.5),
        )
        self.block3 = nn.Sequential(
            conv_bn_relu(base_ch * 2, base_ch * 4),
            conv_bn_relu(base_ch * 4, base_ch * 4),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )
        self.block4 = nn.Sequential(
            conv_bn_relu(base_ch * 4, base_ch * 8),
            conv_bn_relu(base_ch * 8, base_ch * 8),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        feat = base_ch * 8
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        return self.head(x)

# 2. DEFINE SYNTHETIC WAFER GENERATOR
def generate_synthetic_wafer(pattern_type="Scratch"):
    h, w = 64, 64
    grid = np.zeros((h, w), dtype=np.float32)
    cy, cx = 31.5, 31.5
    r_wafer = 28.0
    y, x = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((y - cy)**2 + (x - cx)**2)
    wafer_mask = dist_from_center <= r_wafer
    
    # Base silicon wafer background
    grid[wafer_mask] = 0.05
    
    if pattern_type == "Scratch":
        # Curved scratch line
        for t in np.linspace(0, 1, 100):
            px = int(18 + 25 * t + 3 * np.sin(t * 12))
            py = int(15 + 32 * t)
            if 0 <= px < w and 0 <= py < h:
                grid[py, px] = 0.95
    elif pattern_type == "Edge-Ring":
        ring_mask = (dist_from_center >= 25.0) & (dist_from_center <= 28.0)
        grid[ring_mask] = 0.95
    elif pattern_type == "Center":
        center_cluster = dist_from_center <= 7.0
        grid[center_cluster] = 0.95
    else:
        # Localized clump
        loc_cluster = (np.sqrt((y - 20)**2 + (x - 45)**2) <= 6.0) & wafer_mask
        grid[loc_cluster] = 0.95
        
    # Add slight random noise inside wafer
    noise_mask = wafer_mask & (np.random.rand(h, w) < 0.02)
    grid[noise_mask] = 0.8
    return grid

# 3. INITIALIZE DEVICE & MODEL
CLASS_NAMES = ["Center", "Donut", "Edge-Loc", "Edge-Ring", "Local", "Near-full", "Random", "Scratch"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WaferCNN(in_ch=1, num_classes=8, base_ch=32, dropout=0.3)

if os.path.exists("best_wafercnn.pt"):
    state = torch.load("best_wafercnn.pt", map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state)
    print("Successfully loaded model from best_wafercnn.pt")
else:
    print("Warning: best_wafercnn.pt not found. Running with randomized weights.")

model.to(device).eval()

# 4. RUN INFERENCE & COMPUTE SALIENCY MAP
test_class = "Scratch"  # Options: Scratch, Edge-Ring, Center, Local
wafer_map = generate_synthetic_wafer(test_class)

input_tensor = torch.from_numpy(wafer_map)[None, None, ...].to(device)
input_tensor.requires_grad_()

with torch.enable_grad():
    logits = model(input_tensor)
    probs = F.softmax(logits, dim=1).squeeze(0)
    pred_idx = torch.argmax(probs).item()
    detected_class = CLASS_NAMES[pred_idx]
    detected_confidence = float(probs[pred_idx])
    
    # Backpropagation to input for saliency map
    score = logits[0, pred_idx]
    model.zero_grad()
    score.backward()
    
    saliency = input_tensor.grad.data.abs().squeeze().cpu().numpy()
    lo, hi = float(saliency.min()), float(saliency.max())
    if hi > lo:
        saliency = (saliency - lo) / (hi - lo)
    else:
        saliency = np.zeros_like(saliency)

print(f"✅ Defect Detected: {detected_class} (Confidence: {detected_confidence*100:.1f}%)")
print("Generating saliency map visualization...")

# 5. PLOT RESULTS
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(wafer_map, cmap="gray", origin="lower")
axes[0].set_title(f"Synthetic Wafer Map ({test_class} Pattern)")
axes[0].axis("off")

im = axes[1].imshow(saliency, cmap="jet", origin="lower")
axes[1].set_title(f"Saliency Map (Attribution for: {detected_class})")
axes[1].axis("off")

plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


Loading image and running CNN inference...
✅ Defect Detected: Scratch (Confidence: 92.5%)
Passing data to RCA module...


In [12]:
from IPython.display import display, HTML

# 1. THE DATA DICTIONARIES
REFERENCES = {
    "R1": "Wu, M.-J., Jang, J.-S. R., & Chen, J.-L. (2015). Wafer Map Failure Pattern Recognition and Similarity Ranking for Large-Scale Data Sets. IEEE Trans. Semiconductor Manufacturing, 28(1), 1-12. DOI:10.1109/TSM.2014.2364237",
    "R2": "Piao, M., Jin, C. H., Lee, J. Y., & Byun, J.-Y. (2018). Decision Tree Ensemble-Based Wafer Map Failure Pattern Recognition Based on Radon Transform-Based Features. IEEE Trans. Semiconductor Manufacturing, 31(2), 250-257.",
    "R3": "Wang, R., & Chen, N. (2019). Wafer Map Defect Pattern Recognition Using Rotation-Invariant Features. IEEE Trans. Semiconductor Manufacturing, 32(4), 596-604.",
    "R4": "Adly, F., Yoo, P. D., Muhaidat, S., Al-Hammadi, Y., Lee, U., & Ismail, M. (2015). Randomized General Regression Network for Identification of Defect Patterns in Semiconductor Wafer Maps. IEEE Trans. Semiconductor Manufacturing, 28(2), 145-152.",
    "R5": "Xie, L., Huang, R., Gu, N., & Cao, Z. (2014). A novel defect detection and identification method in optical inspection. Neural Computing and Applications, 24, 1953-1962.",
    "R6": "May, G. S., & Spanos, C. J. (2006). Fundamentals of Semiconductor Manufacturing and Process Control. Wiley-IEEE Press.",
    "R7": "Wolf, S., & Tauber, R. N. (2000). Silicon Processing for the VLSI Era, Vol. 1: Process Technology (2nd ed.). Lattice Press.",
    "R8": "Lieberman, M. A., & Lichtenberg, A. J. (2005). Principles of Plasma Discharges and Materials Processing (2nd ed.). Wiley-Interscience.",
    "R9": "Steigerwald, J. M., Murarka, S. P., & Gutmann, R. J. (1997). Chemical Mechanical Planarization of Microelectronic Materials. Wiley-VCH.",
    "R10": "Hansen, M. H., Nair, V. N., & Friedman, D. J. (1997). Monitoring Wafer Map Data from IC Fabrication Processes for Spatially Clustered Defects. Technometrics, 39(3), 241-253."
}

RCA_KB = {
    "Center": {"description": "Failing dies concentrated in a compact cluster at the wafer center.", "evidence": [["Deposition (CVD/PVD)", "Center-thick/center-thin film nonuniformity", "Center gas loading / showerhead profile", "Medium-High", "R4"], ["Photoresist spin-coat", "Dispense/spread nonuniformity", "Dispense volume & spread program", "Medium", "R6"], ["CMP", "Center over/under-polish", "Polish-head center-zone pressure", "Medium", "R6"], ["RTP/Anneal", "Center-to-edge thermal gradient", "Chuck backside cooling / thermal contact", "Medium", "R6"]], "investigations": ["Verify deposition center-to-edge uniformity", "Review spin-coat dispense & spread recipe", "Check chuck backside He cooling / thermal contact", "Review CMP head center-zone pressure"], "corrective": ["Tune deposition showerhead / gas loading", "Adjust spin-coat dispense program", "Rebalance CMP zonal pressure", "Restore chuck thermal contact"], "evidence_strength": "Medium-High"},
    "Donut": {"description": "Annular band of failing dies at mid-radius; passing center and passing outer edge.", "evidence": [["Photoresist spin-coat", "Radial nonuniformity / solvent-evaporation front", "Spin-speed ramp & bowl exhaust", "Medium", "R6"], ["Deposition", "Annular film-thickness band", "Radial thickness interference", "Medium", "R7"], ["CMP", "Annular (zonal) polish nonuniformity", "Retaining-ring / zonal pressure", "Medium", "R9"], ["Etch", "Radial / standing-wave effect", "Radial gas or temperature band", "Low", "R6"]], "investigations": ["Review spin-speed profile & bowl exhaust", "Inspect CMP retaining ring & zonal pressure", "Check radial gas/temperature uniformity", "Review bake-plate uniformity"], "corrective": ["Adjust spin-speed ramp", "Replace/retune CMP retaining ring", "Rebalance radial gas/temperature", "Correct bake-plate profile"], "evidence_strength": "Medium"},
    "Edge-Loc": {"description": "Failing dies in a localized arc at the wafer edge (partial edge segment).", "evidence": [["Etch / Plasma", "Localized edge plasma nonuniformity", "Localized focus-ring wear", "Medium-High", "R8"], ["Lithography", "Edge focus / exposure error", "Edge dose / focus offset", "Medium", "R3"], ["Edge handling / clamp", "Edge clamp / contact damage", "Damaged clamp pin / contact point", "Medium", "R6"], ["Edge-bead removal (EBR)", "EBR nozzle misalignment", "EBR dispense alignment", "Medium", "R6"]], "investigations": ["Inspect focus ring for localized wear", "Check edge exposure focus/dose", "Inspect wafer clamp pins / contact points", "Verify edge-bead-removal alignment"], "corrective": ["Replace/rotate focus ring", "Recalibrate edge exposure", "Replace damaged clamp pin", "Realign EBR nozzle"], "evidence_strength": "Medium-High"},
    "Edge-Ring": {"description": "Failing dies forming a ring around the entire wafer edge.", "evidence": [["Etch / Plasma", "Edge plasma / sheath nonuniformity", "RF / edge field nonuniformity", "High", "R8"], ["Etch / Plasma", "Focus-ring erosion", "Focus-ring wear", "High", "R6"], ["Etch / Plasma", "Edge gas-distribution effect", "Edge gas flow / chamber edge purge", "Medium", "R6"], ["RTP / Anneal", "Radial thermal gradient during RTA", "Anneal radial temperature profile", "Medium", "R5"]], "investigations": ["Inspect / replace focus ring", "Verify RF power stability", "Check edge gas-flow uniformity & edge purge", "Review RTA radial temperature profile"], "corrective": ["Replace focus ring", "Stabilize RF power delivery", "Rebalance edge gas flow", "Correct RTA radial profile"], "evidence_strength": "High"},
    "Loc": {"description": "A localized cluster of failing dies on the wafer interior (not centered, not a full edge feature).", "evidence": [["Lithography (reticle)", "Reticle / mask defect (repeating field)", "Particle on reticle / pellicle", "Medium", "R6"], ["Deposition", "Localized deposition anomaly", "Chamber flaking / spit", "Medium", "R6"], ["Handling", "Localized handling contact", "Contact-point contamination", "Low", "R10"]], "investigations": ["Reticle / pellicle inspection (check for repeating field signature)", "Particle-source / chamber-flaking check", "Cross-tool commonality analysis"], "corrective": ["Clean / replace reticle or pellicle", "Perform chamber clean (flaking)", "Remediate handling contact point"], "evidence_strength": "Medium"},
    "Near-full": {"description": "Failing dies cover most of the wafer.", "evidence": [["Multiple modules (gross)", "Gross process excursion / missed step", "Missed or incorrect process step", "Medium", "R6"], ["Lithography", "Gross dose / focus error", "Recipe error", "Medium", "R6"], ["Test / Probe", "Probe / contact failure (test artifact)", "Probe-card / contact integrity", "Medium", "R1"]], "investigations": ["Audit process log for missed / incorrect step", "Verify recipe vs golden recipe", "Check probe-card / contact integrity (rule out test artifact)", "Gross-defect inspection"], "corrective": ["Reprocess / correct the missed step", "Restore golden recipe", "Service probe card", "Contamination remediation"], "evidence_strength": "Medium"},
    "Random": {"description": "Failing dies scattered across the wafer with no coherent spatial structure.", "evidence": [["Line-wide (ambient)", "Random particulate contamination", "Cleanroom particle excursion", "Medium-High", "R6"], ["Line-wide", "Background killer-defect density (D0)", "Random material defects", "Medium", "R1"], ["Facilities", "Airborne / chemical contamination", "Filter / FFU degradation", "Medium", "R6"]], "investigations": ["Cleanroom particle audit & FFU/filter check", "Defect-density (D0) monitoring & trend", "Chemical / airborne contamination review (treat as line-wide, not single-tool)"], "corrective": ["Replace / service FFU & filters", "Drive D0 reduction program", "Remediate chemical/airborne source"], "evidence_strength": "Medium-High"},
    "Scratch": {"description": "Failing dies along a line or curvilinear track.", "evidence": [["CMP", "Mechanical scratch (pad / slurry agglomerate)", "Pad conditioning / large slurry particle", "High", "R9"], ["Handling / robotics", "Robot end-effector abrasion", "End-effector contact", "High", "R6"], ["Handling", "Carrier / cassette scrape", "Cassette contact point", "Medium", "R6"]], "investigations": ["Inspect CMP pad & slurry filtration", "Check robot end-effector & handling path", "Inspect wafer carrier / cassette for contact points"], "corrective": ["Recondition / replace CMP pad; improve slurry filtration", "Service / replace end-effector", "Repair cassette contact point"], "evidence_strength": "High"},
    "none": {"description": "No defect signature detected (clean wafer).", "evidence": [], "investigations": [], "corrective": [], "evidence_strength": "N/A"}
}

# 2. THE DASHBOARD BUILDER
def generate_rca_dashboard(defect_class, confidence_score):
    """Processes the inputs from Cell 1 and renders the complete UI dashboard."""

    # Fetch data for the specific defect
    v = RCA_KB.get(defect_class, {"description": "Unknown", "evidence": [], "investigations": [], "corrective": [], "evidence_strength": "N/A"})

    # Format lists into HTML elements
    mod_html = "".join([f"<li>{m}</li>" for m in list(dict.fromkeys(r[0] for r in v["evidence"]))])
    chk_html = "".join([f"<li>{c}</li>" for c in v["investigations"]])
    action_html = "".join([f"<li>{c}</li>" for c in v["corrective"]])

    # Build Evidence Table
    rows = "".join([f"<tr><td>{r[0]}</td><td>{r[1]}</td><td><span class='badge {r[3].lower()}'>{r[3]}</span></td><td>[{r[4]}]</td></tr>" for r in v["evidence"]])

    # Dynamically extract and build the References list specific to this defect
    used_ref_keys = list(dict.fromkeys([r[4] for r in v["evidence"]]))
    ref_html = "".join([f"<li style='margin-bottom: 8px;'><strong>[{k}]</strong> {REFERENCES.get(k, 'Reference not found.')}</li>" for k in used_ref_keys])

    # CSS and HTML Structure
    html = f"""
    <style>
        .rca-card {{ font-family: 'Segoe UI', Tahoma, sans-serif; max-width: 950px; background: #fff; border: 1px solid #e1e4e8; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); padding: 24px; margin-top: 10px; color: #24292e; }}
        .rca-header {{ display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #f0f3f6; padding-bottom: 16px; margin-bottom: 20px; }}
        .rca-header h2 {{ margin: 0; color: #0366d6; font-size: 26px; }}
        .rca-metric .value {{ font-size: 22px; font-weight: bold; color: #28a745; text-align: right; }}
        .rca-grid {{ display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 20px; margin-bottom: 24px; }}
        .rca-section {{ background: #f6f8fa; padding: 16px; border-radius: 6px; }}
        .rca-section h4 {{ margin: 0 0 12px 0; color: #24292e; border-bottom: 1px solid #e1e4e8; padding-bottom: 8px; font-size: 14px; text-transform: uppercase; }}
        .rca-section ul {{ margin: 0; padding-left: 20px; font-size: 13.5px; line-height: 1.5; }}
        .rca-table {{ width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 13.5px; }}
        .rca-table th, .rca-table td {{ padding: 10px; text-align: left; border-bottom: 1px solid #e1e4e8; }}
        .badge {{ padding: 3px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase; }}
        .badge.high {{ background: #ffdce0; color: #cb2431; }} .badge.medium-high {{ background: #fff5b1; color: #b08800; }}
        .badge.medium {{ background: #dbedff; color: #0366d6; }} .badge.low {{ background: #f1f8ff; color: #586069; }}
        .ref-section {{ background: white; border: 1px solid #e1e4e8; padding: 16px; border-radius: 6px; margin-top: 24px; }}
        .ref-section ul {{ list-style-type: none; padding-left: 0; font-size: 12.5px; color: #586069; }}
    </style>

    <div class="rca-card">
        <div class="rca-header">
            <div>
                <h2>Defect Detected: {defect_class}</h2>
                <div style="font-size:14px; color:#586069; margin-top: 4px;">{v["description"]}</div>
            </div>
            <div class="rca-metric">
                <div class="value">{confidence_score*100:.1f}%</div>
                <div style="font-size:13px; color:#586069;">Model Confidence</div>
            </div>
        </div>

        <div class="rca-grid">
            <div class="rca-section"><h4>Likely Process Modules</h4><ul>{mod_html}</ul></div>
            <div class="rca-section"><h4>Recommended Investigations</h4><ul>{chk_html}</ul></div>
            <div class="rca-section"><h4>Corrective Actions</h4><ul>{action_html}</ul></div>
        </div>

        <div class="rca-section" style="background:white; border:1px solid #e1e4e8;">
            <h4>Literature Evidence Traceability (Strength: {v["evidence_strength"]})</h4>
            <table class="rca-table">
                <thead><tr><th>Process Module</th><th>Physical Mechanism</th><th>Evidence</th><th>Citation</th></tr></thead>
                <tbody>{rows}</tbody>
            </table>
        </div>

        <div class="ref-section">
            <h4 style="margin: 0 0 12px 0; color: #24292e; border-bottom: 1px solid #e1e4e8; padding-bottom: 8px; font-size: 14px; text-transform: uppercase;">Literature References</h4>
            <ul>{ref_html}</ul>
        </div>
    </div>
    """

    display(HTML(html))

# ---------------------------------------------------------
# EXECUTE: Call the dashboard builder using the variables
# generated from Cell 1
# ---------------------------------------------------------
generate_rca_dashboard(detected_class, detected_confidence)

Process Module,Physical Mechanism,Evidence,Citation
CMP,Mechanical scratch (pad / slurry agglomerate),High,[R9]
Handling / robotics,Robot end-effector abrasion,High,[R6]
Handling,Carrier / cassette scrape,Medium,[R6]
